# Run inference a pre-trained model

## Load a pre-trained model from checkpoints (previous notebook)

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

import orbax
from orbax import checkpoint
from jax.sharding import SingleDeviceSharding

from src.model.model import NanoLLM
from src.inference.generate import generate_text
from src.config import ModelConfig, TokenizerConfig, InferenceConfig
from src.paths import CHECKPOINTS_DIR

In [ ]:
# import model and default configuration
model_config = ModelConfig()
model = NanoLLM(model_config)

In [ ]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

checkpoint_path = CHECKPOINTS_DIR / "nano_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [ ]:
# TODO: extract to these to src directory
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

In [ ]:
# Update the model with the loaded checkpoints
nnx.update(model,restored_state)

## Run inference on the pre-trained model

In [ ]:
# tokenizer and delimiter are stored together in TokenizerConfig because they are both essential properties that describe how a model was trained.
# This pair should not be coupled directly to the model itself because they might change if we retrain the model.
tokenizer_config = TokenizerConfig(delimiter="<|endoftext|>", name="gpt2")
tokenizer = tokenizer_config.tokenizer

# Define method for creating a story
def create_story(story_prompt, max_new_tokens, temperature):

    # Convert the text prompt to token IDs using the tokenizer
    start_tokens = tokenizer.encode(story_prompt)

    # Ensure max_new_tokens is an integer
    # Underlying inference functions require that max_new_tokens is an integer, 
    # but some callers (e.g. Gradio sliders) return floats for this value
    max_new_tokens = int(max_new_tokens)
    
    inference_config = InferenceConfig(max_new_tokens=max_new_tokens, temperature=temperature)

    return generate_text(model, tokenizer_config, inference_config, start_tokens)

In [ ]:
# Call the method to exercise inference
# The result at this time is super unsatisfactory because the model was not trained very long 
# and the inference method is not optimized.
create_story("I was eating a bowl of raspberries when I noticed that", 30, 0.2)

## Build UI for running inference

In [ ]:
import gradio as gr

# define demo
demo = gr.Interface(

    # the inference method developed previously
    fn=create_story,

    inputs=[
        gr.Textbox(label="Starter Prompt"),   
        gr.Slider(minimum=0, maximum=200, value=30, step=1, label="Max Tokens"),
        gr.Slider(minimum=0, maximum=1.0, value=0.2, step=0.01, label="Temperature")
    ],
    
    outputs=["text"]
)

In [ ]:
# launch demo; share=True will provide the local url
demo.launch(share=True)